# 4.3 Data Preprocessing and Data Representation

**News Simplification Assistant — Data Preparation Notebook**

This notebook prepares the XSum dataset for prompting `facebook/opt-125m`.
Each numbered section below corresponds to one step of the preprocessing
pipeline described in Section 4.3 of the report:

1. Download the XSum dataset and save a raw copy to `data/raw`
2. Load selected XSum articles and reference summaries
3. Remove empty or duplicated articles
4. Clean newline characters, extra spaces and unnecessary symbols
5. Limit articles that exceed the model input length
6. Insert the cleaned article into the fixed prompt template
7. Tokenise the prompt using the OPT tokenizer before generation

If `data/raw` is empty, the notebook downloads XSum from the Hugging Face
Hub first. The final processed dataset (cleaned articles + prompts + token
ids) is written to `data/processed` in the last cell.


## 0. Setup — Imports and Paths

In [1]:
import os
import re
import json
import glob
import random

import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

random.seed(42)

print(f"Raw data directory:       {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")


Raw data directory:       data/raw
Processed data directory: data/processed


## 1. Download the XSum Dataset to `data/raw`

`data/raw` does not contain the XSum dataset yet, so this step downloads
it from the Hugging Face Hub (`EdinburghNLP/xsum`) and writes a raw JSONL
copy to `data/raw/xsum_raw.jsonl`.

- If a raw file already exists, the download is skipped (safe to re-run).
- `XSUM_SAMPLE_SIZE` controls how many articles are pulled for this
  project so downstream steps stay fast; set it to `None` to download
  the full dataset (~226k train articles).
- Requires internet access to `huggingface.co`. If this environment has
  no internet access, run this cell once on a machine that does, then
  copy the resulting `data/raw/xsum_raw.jsonl` file over.


In [2]:
from datasets import load_dataset

XSUM_SAMPLE_SIZE = 1001
XSUM_SEED = 42



os.makedirs(RAW_DIR, exist_ok=True)
output_path = os.path.join(RAW_DIR, "xsum_raw.jsonl")

if os.path.exists(output_path):
    print(f"Raw XSum file already exists at '{output_path}', skipping download.")
    
else:
    print("Downloading XSum dataset from the Hugging Face Hub (EdinburghNLP/xsum)...")
    dataset = load_dataset("EdinburghNLP/xsum", split="train")

    if XSUM_SAMPLE_SIZE is not None and XSUM_SAMPLE_SIZE < len(dataset):
        dataset = dataset.shuffle(seed=XSUM_SEED).select(range(XSUM_SAMPLE_SIZE))

        with open(output_path, "w", encoding="utf-8") as f:
            for row in dataset:
                record = {
                    "id": row.get("id"),
                    "document": row.get("document"),
                    "summary": row.get("summary"),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"Saved {len(dataset)} raw XSum articles to '{output_path}'")

Raw XSum file already exists at 'data/raw\xsum_raw.jsonl', skipping download.


## 2. Load Selected XSum Articles and Reference Summaries

The raw XSum data downloaded in Section 1 is read back from `data/raw`.
This loader supports a few common formats so it works regardless of how
the raw dump was saved:

- A HuggingFace `datasets` folder saved with `save_to_disk()`
- A single `.json` / `.jsonl` file (list of records or one record per line)
- A `.csv` file

Each record is expected to contain an **article** field and a **reference
summary** field (XSum uses `document` and `summary`).


In [3]:
json_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.jsonl")))
if json_files:
    records = []
    for fp in json_files:
        with open(fp, "r", encoding="utf-8") as f:
            if fp.endswith(".jsonl"):
                for line in f:
                    line = line.strip()
                    if line:
                        records.append(json.loads(line))
            else:
                data = json.load(f)
                records.extend(data if isinstance(data, list) else [data])

df = pd.DataFrame(records)

rename_map = {
    "document": "article",
    "summary": "summary",
    "id": "article_id",
}
raw_df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print(f"Loaded {len(raw_df)} raw articles from '{RAW_DIR}'")
raw_df.head()


Loaded 1001 raw articles from 'data/raw'


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 3. Remove Empty or Duplicated Articles

Records with an empty article/summary, or with an article that is a
duplicate of one already seen, are dropped.


In [4]:
def remove_empty_and_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)

    # Drop missing values
    df = df.dropna(subset=["article", "summary"])

    # Drop empty / whitespace-only strings
    df = df[df["article"].str.strip().astype(bool)]
    df = df[df["summary"].str.strip().astype(bool)]

    # Drop exact duplicate articles, keep the first occurrence
    df = df.drop_duplicates(subset=["article"], keep="first")

    df = df.reset_index(drop=True)

    after = len(df)
    print(f"Removed {before - after} empty/duplicate articles ({before} -> {after})")
    return df


dedup_df = remove_empty_and_duplicates(raw_df)
dedup_df.head()


Removed 1 empty/duplicate articles (1001 -> 1000)


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 4. Clean Newline Characters, Extra Spaces and Unnecessary Symbols

Basic text normalisation applied to both the article and the reference
summary:

- Collapse `\n`, `\r`, `\t` into single spaces
- Collapse repeated whitespace into a single space
- Strip stray control characters and unusual unicode artefacts
- Trim leading/trailing whitespace


In [5]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalise newlines / tabs to spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)

    # Remove stray repeated punctuation/symbols e.g. "----", "****"
    text = re.sub(r"[-*_=~]{3,}", " ", text)

    # Collapse multiple spaces into one
    text = re.sub(r"\s+", " ", text)

    return text.strip()


clean_df = dedup_df.copy()
clean_df["article"] = clean_df["article"].apply(clean_text)
clean_df["summary"] = clean_df["summary"].apply(clean_text)

# Cleaning can occasionally expose new empty strings - drop those too
clean_df = clean_df[clean_df["article"].astype(bool)].reset_index(drop=True)

print(f"{len(clean_df)} articles remaining after cleaning")
clean_df.head()


1000 articles remaining after cleaning


,article_id,article,summary
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...


## 5. Insert the Cleaned Article into the Fixed Prompt Template

The same fixed instruction template is used for every article, asking
the model to produce four sections: **Summary**, **Key Points**,
**Vocabulary**, and **Quiz Questions**.


In [6]:
PROMPT_TEMPLATE = """Simplify the following news article for students using clear and simple English. 
Generate the following sections:
(1) Summary: a short, simplified summary of the article.
(2) Key Points: 3-5 bullet points with the most important facts.
(3) Vocabulary: 3-5 specialized words from the article with simple definitions.
(4) Quiz Questions: 2-3 questions based on the article.
Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article. Ensure all generated content is factually consistent with the original article.
Article: 
{article}
"""

def build_prompt(article: str) -> str:
    return PROMPT_TEMPLATE.format(article=article)


prompt_df = clean_df.copy()
prompt_df["prompt"] = prompt_df["article"].apply(build_prompt)

print(prompt_df.loc[0, "prompt"])


Simplify the following news article for students using clear and simple English. 
Generate the following sections:
(1) Summary: a short, simplified summary of the article.
(2) Key Points: 3-5 bullet points with the most important facts.
(3) Vocabulary: 3-5 specialized words from the article with simple definitions.
(4) Quiz Questions: 2-3 questions based on the article.
Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article. Ensure all generated content is factually consistent with the original article.
Article: 
In Wales, councils are responsible for funding and overseeing schools. But in England, Mr Osborne's plan will mean local authorities will cease to have a role in providing education. Academies are directly funded by central government and head teachers have more freedom over admissions and to change the way the school works. It is a significant development in the continued divergen

## 6. Tokenise the Prompt Using the OPT Tokenizer

Each prompt is converted into token ids using the tokenizer that ships
with `facebook/opt-125m`. This is the **data representation** step:
the article/prompt text becomes the numeric input representation the
model consumes, and generated output tokens will later be decoded back
into text using this same tokenizer.

Any prompt whose token length still exceeds the model's context window
after word-level truncation (Section 4) is truncated again here at the
token level, to guarantee correctness.


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/opt-125m"
MAX_MODEL_TOKENS = 2048
MAX_PROMPT_TOKENS = 900

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_prompt(prompt: str):
    encoded = tokenizer(
        prompt,
        truncation=False
    )
    input_ids = encoded["input_ids"]
    return input_ids, len(input_ids)


token_df = prompt_df.copy()
token_df["prompt_word_count"] = token_df["prompt"].str.split(" ").apply(len)
token_results = token_df["prompt"].apply(tokenize_prompt)
token_df["input_ids"] = token_results.apply(lambda x: x[0])
token_df["prompt_token_length"] = token_results.apply(lambda x: x[1])

print(token_df[["article_id", "prompt_word_count", "prompt_token_length"]].describe(include="all"))
token_df.head()


       article_id  prompt_word_count  prompt_token_length
count        1000        1000.000000          1000.000000
unique       1000                NaN                  NaN
top      35821725                NaN                  NaN
freq            1                NaN                  NaN
mean          NaN         457.124000           608.349000
std           NaN         294.629957           366.182426
min           NaN         111.000000           173.000000
25%           NaN         260.750000           360.000000
50%           NaN         379.000000           512.500000
75%           NaN         579.250000           753.000000
max           NaN        3572.000000          4421.000000


,article_id,article,summary,prompt,prompt_word_count,input_ids,prompt_token_length
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...,Simplify the following news article for studen...,633,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ...",778
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...,Simplify the following news article for studen...,295,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ...",412
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal...",Simplify the following news article for studen...,755,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ...",995
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...,Simplify the following news article for studen...,647,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ...",828
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...,Simplify the following news article for studen...,422,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ...",546


## 7. Assemble the Final Processed Dataset

Combine the cleaned article, reference summary, fixed prompt, and
tokenised representation into the final structured table that will be
saved for the fine-tuning / generation stage.


In [8]:
final_df = token_df[[
    "article_id",
    "article",
    "summary",
    "prompt",
    "prompt_word_count",
    "prompt_token_length",
    "input_ids",
]].reset_index(drop=True)

print(f"Final processed dataset: {len(final_df)} rows")
final_df.head()


Final processed dataset: 1000 rows


,article_id,article,summary,prompt,prompt_word_count,prompt_token_length,input_ids
0,35821725,"In Wales, councils are responsible for funding...",As Chancellor George Osborne announced all Eng...,Simplify the following news article for studen...,633,778,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ..."
1,40302059,"Up to 100,000 youngsters will be eligible for ...",Discounted bus tickets for 16 to 18 year olds ...,Simplify the following news article for studen...,295,412,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ..."
2,36192574,Middlesbrough and Brighton face each other on ...,"""The winner takes it all, the loser has to fal...",Simplify the following news article for studen...,755,995,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ..."
3,32831331,"Money allocated to children's centres, which p...",Support for vulnerable children in England is ...,Simplify the following news article for studen...,647,828,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ..."
4,26781682,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...,Simplify the following news article for studen...,422,546,"[2, 29656, 2911, 4591, 5, 511, 340, 1566, 13, ..."


## 8. Save Processed Data to `data/processed`

The final dataset is saved as JSONL (one record per line) so it can be
loaded lazily for the generation/fine-tuning stage without pulling the
whole dataset into memory.


In [9]:
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "xsum_processed.jsonl")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in final_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(final_df)} processed records to '{OUTPUT_PATH}'")


Saved 1000 processed records to 'data/processed\xsum_processed.jsonl'
